In [0]:
# ==============================================================================
# DOCUMENTAÇÃO DO NOTEBOOK
# Nome: 03_gold_kpis_diarios
# Camada: Gold (Agregações, KPIs e Consumo)
# Descrição: 
#   Este notebook lê as tabelas consolidadas e limpas da camada Silver 
#   (silver_viagens, silver_caronas e silver_abastecimento), realiza o cruzamento 
#   (JOIN) por data e aplica regras de enriquecimento de negócio (como cálculo de 
#   lucro líquido diário e custo por quilômetro rodado - CPKM). 
#   Por fim, persiste os dados na tabela Delta Gold para consumo analítico e 
#   visualizações no Databricks.
# ==============================================================================

from pyspark.sql import functions as F

catalogo = "will"
esquema = "i30"

# Lendo as tabelas da camada Silver
df_viagens = spark.table(f"{catalogo}.{esquema}.silver_viagens")
df_caronas = spark.table(f"{catalogo}.{esquema}.silver_caronas")
df_abastecimento = spark.table(f"{catalogo}.{esquema}.silver_abastecimento")

# 1. Agrupando e consolidando por DATA
df_gold = (
    df_viagens
    .join(df_caronas, "DATA", "left")
    .join(df_abastecimento, "DATA", "left")
    .select(
        df_viagens["DATA"],
        df_viagens["KM_RODADAS"],
        df_viagens["PAINEL_L_100KM"],
        df_viagens["VELOC_MEDIA_KM_H"],
        F.coalesce(df_caronas["TOTAL_PASSAGEIROS"], F.lit(0)).alias("TOTAL_PASSAGEIROS"),
        F.coalesce(df_caronas["TOTAL_GANHO_REAIS"], F.lit(0.0)).alias("TOTAL_GANHO_REAIS"),
        F.coalesce(df_abastecimento["POSTO_VALOR_TOTAL"], F.lit(0.0)).alias("CUSTO_ABASTECIMENTO_REAIS"),
        F.coalesce(df_abastecimento["LITROS_ABASTECIDOS"], F.lit(0.0)).alias("LITROS_ABASTECIDOS")
    )
    # 2. Enriquecimento de Negócio na Gold (KPIs)
    .withColumn(
        "LUCRO_LIQUIDO_DIA", 
        F.col("TOTAL_GANHO_REAIS") - F.col("CUSTO_ABASTECIMENTO_REAIS")
    )
    .withColumn(
        "CUSTO_POR_KM",
        F.when(F.col("KM_RODADAS") > 0, F.col("CUSTO_ABASTECIMENTO_REAIS") / F.col("KM_RODADAS"))
         .otherwise(0.0)
    )
    .orderBy("DATA")
)

# 3. Gravando a tabela Gold em Delta Lake
tabela_gold = f"{catalogo}.{esquema}.gold_kpis_diarios"
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_gold)

print(f"Camada Gold gerada com sucesso: {tabela_gold}")
display(spark.table(tabela_gold))